# Phase 3: Hybrid GraphRAG Search Engine (From Scratch)

이 노트북에서는 LangChain의 블랙박스 리트리버를 쓰지 않고, **직접 OpenAI 임베딩을 생성하여 Neo4j에 주입하고, 순수 Cypher 쿼리를 통해 하이브리드 검색(Vector + Graph)**을 구현합니다.

In [ ]:
!pip install -qU openai neo4j python-dotenv

## 1. 환경 변수 및 클라이언트 세팅

In [ ]:
import os
import sys
from dotenv import load_dotenv
from neo4j import GraphDatabase
from openai import OpenAI

sys.path.append('../../')
load_dotenv('../../.env')

URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
AUTH = (os.getenv("NEO4J_USERNAME", "neo4j"), os.getenv("NEO4J_PASSWORD", "graphrag2026"))

driver = GraphDatabase.driver(URI, auth=AUTH)
client = OpenAI() # OPENAI_API_KEY 환경변수 자동 인식
print("Clients initialized!")

## 2. 노드 임베딩 주입 및 Vector Index 생성
그래프에 있는 모든 노드를 가져와 OpenAI API로 1536차원 벡터(임베딩)로 변환한 뒤 다시 노드에 저장합니다. 이후 빠른 검색을 위해 Neo4j Vector Index를 생성합니다.

In [ ]:
def setup_vector_embeddings(tx):
    # 1. 모든 노드에 공통 'Entity' 라벨 부여 (인덱싱을 위함)
    tx.run("MATCH (n) SET n:Entity")
    
    # 2. 임베딩이 없는 노드들 가져오기
    result = tx.run("MATCH (n:Entity) WHERE n.embedding IS NULL RETURN elementId(n) AS internal_id, n.id AS text")
    nodes = [record for record in result]
    
    if nodes:
        print(f"Generating embeddings for {len(nodes)} nodes...")
        texts = [n['text'] for n in nodes]
        # OpenAI API 호출 (text-embedding-3-small)
        response = client.embeddings.create(input=texts, model="text-embedding-3-small")
        
        # 3. 임베딩 벡터를 노드에 저장
        for i, node in enumerate(nodes):
            embedding = response.data[i].embedding
            tx.run("MATCH (n) WHERE elementId(n) = $int_id SET n.embedding = $emb", int_id=node['internal_id'], emb=embedding)
        print("Embeddings successfully saved to Neo4j!")

def create_vector_index(tx):
    # 4. Vector Index 생성 (이미 존재하면 무시)
    query_index = """
    CREATE VECTOR INDEX entity_index IF NOT EXISTS
    FOR (n:Entity) ON (n.embedding)
    OPTIONS {indexConfig: {
      `vector.dimensions`: 1536,
      `vector.similarity_function`: 'cosine'
    }}
    """
    tx.run(query_index)
    print("Vector Index 'entity_index' is ready.")

with driver.session() as session:
    session.execute_write(setup_vector_embeddings)
    session.execute_write(create_vector_index)

## 3. 하이브리드 리트리버 (Vector + Graph Search)
질문을 벡터로 변환하여 가장 유사한 시작 노드 3개를 찾고, 그 노드들과 연결된 주변 1-hop 문맥을 모조리 긁어옵니다.

In [ ]:
def hybrid_search(question, top_k=3):
    # 1. 질문 임베딩
    question_embedding = client.embeddings.create(input=question, model="text-embedding-3-small").data[0].embedding
    
    # 2. Cypher: 벡터 검색으로 시작 노드를 찾고, 그래프를 순회하여 문맥 추출
    query = """
    CALL db.index.vector.queryNodes('entity_index', $k, $embedding)
    YIELD node, score
    MATCH (node)-[r]-(neighbor)
    RETURN node.id AS source, type(r) AS relationship, neighbor.id AS target, score
    ORDER BY score DESC
    """
    
    with driver.session() as session:
        results = session.run(query, k=top_k, embedding=question_embedding)
        
        context_list = []
        for record in results:
            # 예: "Tesla -COMPETES_WITH-> Rivian"
            fact = f"{record['source']} -[{record['relationship']}]-> {record['target']}"
            if fact not in context_list:
                context_list.append(fact)
                
    return "\n".join(context_list)

# 테스트
user_question = "What are the main risks associated with Tesla?"
context = hybrid_search(user_question)
print("[Extracted Graph Context]")
print(context)

## 4. LLM 답변 생성 (The Final RAG)
추출한 그래프 문맥을 프롬프트에 담아 최종 답변을 생성합니다.

In [ ]:
def generate_answer(question):
    context = hybrid_search(question)
    
    system_prompt = """
    You are an expert financial graph analyst. 
    Answer the user's question using ONLY the provided knowledge graph context.
    If the context doesn't contain the answer, say 'I do not have enough information'.
    
    [GRAPH CONTEXT]
    {context}
    """
    
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt.format(context=context)},
            {"role": "user", "content": question}
        ],
        temperature=0
    )
    return response.choices[0].message.content

print("\n--- Final Answer ---")
print(generate_answer(user_question))